# Libraries

Before run this notebook you need install the dependencies. See the `requirements.txt` file.


In [1]:
from functions_victor_project import *

import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import matplotlib.pyplot as plt
import pickle
import dill
from scipy.integrate import odeint
from UQpy.distributions import Uniform, Normal, JointIndependent #, Lognormal
from UQpy.distributions.collection.Lognormal import Lognormal
from UQpy.surrogates import *
from sklearn.metrics import mean_squared_error, r2_score

d:\Documentos\ic_victor\myenv\Lib\site-packages\UQpy\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Statistics of the input variables

- $R$: Resistance variable  
- $S$: Demand variable  
- $Z_1$ and $Z_2$: Latent variables  

State Limit Function:

$$
g(R, S, t) = k_{factor}(t) \cdot \frac{R}{Z_1} - S \cdot Z_2
$$

**Table 1:** Moments, distributions, and parameters of the considered variables for the $R$ and $S$ problem.

| Variable | Distribution | Mean | Std. Deviation |  
|----------|--------------|------|----------------|  
| $R$    | Normal     | 5.0  | 0.8            |  
| $S$    | Normal     | 2.0  | 0.6            |  
| $Z_1$ | Lognormal   | 1.0  | 0.028          |  
| $Z_2$ | Lognormal   | 1.0  | 0.096          |  

To simulate $Z_1$ and $Z_2$ we are used `scipy` library. You can see the file `glam_example_1.py` and how to generate state limit results. 

### Transformation to scipy parameters

**Shape parameter (s):**
$$
s = \sqrt{\ln\left(1 + \left(\frac{\sigma}{\mu}\right)^2\right)}
$$

**Scale parameter:**
$$
\text{scale} = \frac{\mu}{\sqrt{1 + \left(\frac{\sigma}{\mu}\right)^2}}
$$

Where:
- $\mu$ = mean of the lognormal distribution  
- $\sigma$ = standard deviation of the lognormal distribution
- $s$ = shape parameter for `scipy.stats.lognorm`
- $\text{scale}$ = scale parameter for `scipy.stats.lognorm`

# Full process for time PCE

KeyboardInterrupt: 

In [ ]:
n_samples = 1000
n_latent_samples = 5000
time = [0, 1, 5, 20, 50, 75, 100]

for k in time:
    
    # training
    r   = Normal(loc = 5., scale=0.8)
    s   = Normal(loc = 2., scale=0.6)
    joint = JointIndependent(marginals=[r, s])
    x = joint.rvs(n_samples)
    y, _ = state_limit_function_time(x, n_latent_samples, t=k)
    max_degree = 3
    polynomial_basis = TotalDegreeBasis(joint, max_degree)
    least_squares = LeastSquareRegression()
    pce_metamodel = PolynomialChaosExpansion(polynomial_basis=polynomial_basis, regression_method=least_squares)
    pce_metamodel.fit(x, y)
    
    # save metamodel
    with open(f'pce_metamodel_{k}.pkl', 'wb') as f:
        dill.dump(pce_metamodel, f)
    
    # validation
    r   = Normal(loc = 5., scale=0.8)
    s   = Normal(loc = 2., scale=0.6)
    joint = JointIndependent(marginals=[r, s])
    x_val = joint.rvs(n_samples)
    y_val_obs, _ = state_limit_function_time(x_val, n_latent_samples, t=k)
    with open(f'pce_metamodel_{k}.pkl', 'rb') as f:
        pce_metamodel_up = pickle.load(f)
    y_val_pre = pce_metamodel_up.predict(x_val)
    y_val_obs_aux = list(y_val_obs[:, 0])
    y_val_pre_aux = list(y_val_pre[:, 0])       
    print(f"time {k} - R2:", r2_score(y_val_obs_aux, y_val_pre_aux))      